# LangGraph Deployment — Shipping Agents to SAP BTP

Builds on **Notebook 1 (Fundamentals)** and **Notebook 2 (Intermediate)**.
This notebook has a **narrative arc**: we build one deployable SAP agent app from scratch, adding a layer in each section.

## What we build

A SAP workflow agent that can search a knowledge base and look up sales orders,  
served via a FastAPI REST API, tested with pytest, and deployed to SAP BTP Cloud Foundry.

```
Section 1 ──► Base agent + test suite
Section 2 ──► Wrap agent in FastAPI (POST /chat, GET /health)
Section 3 ──► Add streaming endpoint (POST /chat/stream, SSE)
Section 4 ──► Wire real SAP AI Core credentials (gen_ai_hub)
Section 5 ──► Add real SAP tools (HANA vector search + OData GET)
Section 6 ──► Package and deploy to Cloud Foundry (manifest.yml, cf push)
Section 7 ──► Extend with MCP tools (langchain-mcp-adapters)
Section 8 ──► Integrate with SAP Joule Studio
```

## Roadmap

| # | Section | Key Concept | SAP Relevance |
|---|---------|-------------|---------------|
| 1 | Testing agents | pytest · node unit tests · graph integration tests · mock LLM | CI habit before CF push |
| 2 | FastAPI serving layer | `POST /chat` · `GET /health` · Pydantic models · TestClient | Standard BTP API pattern (QuickLaunch Part 6) |
| 3 | Streaming | `astream_events` · SSE · `StreamingResponse` | Real-time output for Fiori / BTP Build Apps |
| 4 | AI Core credential wiring | `AICORE_SERVICE_KEY` · `gen_ai_hub` auto-detection | Swap mock LLM for real BTP LLM |
| 5 | SAP workflow tools | HANA vector search · CAP/S4 OData GET (mocked) | The SAP-specific substance |
| 6 | Cloud Foundry deployment | `manifest.yml` · `runtime.txt` · `requirements.txt` · `cf push` | QuickLaunch Part 8 pattern |
| 7 | MCP tools | `langchain-mcp-adapters` · CAP MCP server pattern | SAP tools ecosystem |
| 8 | Joule Studio integration | Architecture · BTP Destination · GA late 2025 | Enterprise SAP embedding |

> **Note on deployability:** all Python code in this notebook is runnable without SAP credentials (mocked).  
> Section 6 shows deployment *templates* — actually running `cf push` requires a BTP account.

---

## Section 1 — Testing Agents

Write tests **before** you deploy. LangGraph agents are easy to test because:
- Tool functions are plain Python — callable directly, no graph needed
- The full graph can be integration-tested with `MemorySaver` (no DB required)
- The LLM can be mocked with `unittest.mock` — avoiding API costs in CI

```
Test pyramid for LangGraph agents:

        ▲  End-to-end (live LLM + real tools)
       ▲▲  Integration (full graph, mock LLM)
      ▲▲▲  Unit (individual node functions)
```

We start by defining the agent we'll use throughout this notebook, then write tests for it.

### 1.1 Base Agent (used across all sections)

In [ ]:
import os
from dotenv import load_dotenv
from typing import TypedDict, Annotated, List, Optional
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from langchain.agents import create_agent

# Load variables from .env in the project root
load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError(
        "OPENAI_API_KEY not found. "
        "Add it to your .env file: OPENAI_API_KEY=sk-..."
    )


# ── Mock SAP tools (upgraded in Section 5) ────────────────────────────────
@tool
def lookup_order(order_id: str) -> str:
    'Look up a SAP sales order by ID. Example: SO-1001'
    orders = {
        'SO-1001': 'Customer: ACME Corp — Amount: 12,500 EUR — Status: Open',
        'SO-1002': 'Customer: SAP SE   — Amount: 87,000 EUR — Status: Delivered',
    }
    return orders.get(order_id, f'Order {order_id} not found.')


@tool
def search_knowledge_base(query: str) -> str:
    'Search the SAP knowledge base for relevant context.'
    if 'order' in query.lower():
        return 'SAP orders use statuses: Open, In Progress, Delivered, Cancelled.'
    if 'ticket' in query.lower():
        return 'SAP tickets are routed by priority: high (SLA 2h), medium (8h), low (24h).'
    return 'No specific SAP context found for this query.'


SAP_TOOLS = [lookup_order, search_knowledge_base]


# ── Build function — the single entry point used across all sections ───────
def build_agent(llm=None, tools=None, checkpointer=None):
    'Compile and return the SAP agent. All parameters are optional.'
    if llm is None:
        llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
    if tools is None:
        tools = SAP_TOOLS
    return create_agent(model=llm, tools=tools, checkpointer=checkpointer)


print('Base agent and tools defined.')
print('Tools:', [t.name for t in SAP_TOOLS])

### 1.2 Unit Tests — Tool Functions

Test each `@tool` function in isolation. No LLM, no graph — just Python.

In [ ]:
# ── Unit tests: tool functions ─────────────────────────────────────────────
# In a real project these go in tests/test_tools.py and run with: pytest tests/

def test_lookup_order_found():
    result = lookup_order.invoke({'order_id': 'SO-1001'})
    assert 'ACME' in result
    assert 'Open' in result
    print('  ✓ test_lookup_order_found')


def test_lookup_order_not_found():
    result = lookup_order.invoke({'order_id': 'SO-9999'})
    assert 'not found' in result.lower()
    print('  ✓ test_lookup_order_not_found')


def test_search_kb_order_context():
    result = search_knowledge_base.invoke({'query': 'sales order status'})
    assert 'Open' in result or 'status' in result.lower()
    print('  ✓ test_search_kb_order_context')


def test_search_kb_ticket_context():
    result = search_knowledge_base.invoke({'query': 'ticket priority SLA'})
    assert 'high' in result.lower() or 'priority' in result.lower()
    print('  ✓ test_search_kb_ticket_context')


def test_search_kb_fallback():
    result = search_knowledge_base.invoke({'query': 'unrelated topic xyz'})
    assert 'No specific' in result
    print('  ✓ test_search_kb_fallback')


print('Unit tests (tool functions):')
test_lookup_order_found()
test_lookup_order_not_found()
test_search_kb_order_context()
test_search_kb_ticket_context()
test_search_kb_fallback()
print('All unit tests passed.')

### 1.3 Integration Tests — Full Graph with Mocked LLM

Test the complete graph without making real API calls. Mock the LLM with `unittest.mock`.

In [ ]:
from unittest.mock import MagicMock
from langgraph.checkpoint.memory import MemorySaver


def make_mock_llm(response_content: str = 'Mocked response.'):
    'Return a mock LLM that always replies with response_content (no tool calls).'
    mock = MagicMock()
    mock.bind_tools.return_value = mock
    mock.invoke.return_value = AIMessage(content=response_content)
    return mock


def test_agent_returns_ai_message():
    agent = build_agent(llm=make_mock_llm('The order is Open.'))
    result = agent.invoke({'messages': [HumanMessage(content='What is SO-1001?')]})
    assert len(result['messages']) > 0
    assert isinstance(result['messages'][-1], AIMessage)
    print('  ✓ test_agent_returns_ai_message')


def test_agent_memory_persists_across_turns():
    checkpointer = MemorySaver()
    agent = build_agent(llm=make_mock_llm('Hello!'), checkpointer=checkpointer)
    config = {'configurable': {'thread_id': 'test-memory-001'}}

    # Turn 1
    agent.invoke({'messages': [HumanMessage(content='Hi, I am Alex.')]}, config=config)
    # Turn 2
    agent.invoke({'messages': [HumanMessage(content='What is my name?')]}, config=config)

    state = agent.get_state(config)
    # Both turns are in state — 2 human + 2 AI messages
    assert len(state.values['messages']) >= 4
    print('  ✓ test_agent_memory_persists_across_turns')


def test_agent_thread_isolation():
    'Two different thread_ids must not share state.'
    checkpointer = MemorySaver()
    agent = build_agent(llm=make_mock_llm('Ok.'), checkpointer=checkpointer)

    cfg_a = {'configurable': {'thread_id': 'thread-a'}}
    cfg_b = {'configurable': {'thread_id': 'thread-b'}}

    agent.invoke({'messages': [HumanMessage(content='Message in thread A')]}, config=cfg_a)

    state_b = agent.get_state(cfg_b)
    # Thread B should have no messages
    assert len(state_b.values.get('messages', [])) == 0
    print('  ✓ test_agent_thread_isolation')


print('Integration tests (full graph, mock LLM):')
test_agent_returns_ai_message()
test_agent_memory_persists_across_turns()
test_agent_thread_isolation()
print('All integration tests passed.')

### 1.4 Running with pytest

In a real project, save tests to `tests/test_agent.py` and run:

```bash
pytest tests/ -v
```

A minimal `tests/test_agent.py` looks like this:

```python
# tests/test_agent.py
import pytest
from unittest.mock import MagicMock
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import MemorySaver
from agent import build_agent, lookup_order   # your module


def test_lookup_order_found():
    assert 'ACME' in lookup_order.invoke({'order_id': 'SO-1001'})


def test_agent_responds():
    mock = MagicMock()
    mock.bind_tools.return_value = mock
    mock.invoke.return_value = AIMessage(content='Done.')
    agent = build_agent(llm=mock)
    result = agent.invoke({'messages': [HumanMessage(content='Hello')]})
    assert isinstance(result['messages'][-1], AIMessage)
```

**Tip for CI on SAP BTP:** add a `pytest` step to your GitHub Actions / Jenkins pipeline before `cf push`.  
Mock the LLM in CI to avoid AI Core costs and rate limits.

---
## Section 2 — FastAPI Serving Layer

The agent runs inside a **FastAPI** app. This is the pattern used in the SAP AI Core  
Agent QuickLaunch series (Part 6) and is the standard way to expose LangGraph agents on BTP.

```
Client
  │
  ├─ POST /chat         ← synchronous: wait for full response
  ├─ POST /chat/stream  ← streaming: tokens arrive as SSE (Section 3)
  └─ GET  /health       ← liveness probe for Cloud Foundry health check
```

### 2.1 The FastAPI App

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel


# ── Request / response models ──────────────────────────────────────────────
class ChatRequest(BaseModel):
    message:   str
    thread_id: Optional[str] = 'default'
    user_id:   Optional[str] = 'anonymous'

class ChatResponse(BaseModel):
    reply:     str
    thread_id: str


# ── App factory ───────────────────────────────────────────────────────────
# Using a factory function (not a module-level app) keeps testing clean:
# each test can get a fresh app with its own in-memory checkpointer.

def create_app(llm=None) -> FastAPI:
    'Build the FastAPI app. Accepts an optional custom LLM for testing.'

    _agent = build_agent(llm=llm, checkpointer=MemorySaver())

    app = FastAPI(title='SAP Agent API', version='1.0.0')

    @app.get('/health')
    def health():
        return {'status': 'ok', 'service': 'sap-agent'}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        config = {'configurable': {'thread_id': req.thread_id, 'user_id': req.user_id}}
        result = _agent.invoke(
            {'messages': [HumanMessage(content=req.message)]},
            config=config,
        )
        return ChatResponse(reply=result['messages'][-1].content, thread_id=req.thread_id)

    return app


# Production app (used when running the server)
app = create_app()

print('FastAPI app created.')
print('Endpoints:')
for route in app.routes:
    if hasattr(route, 'methods'):
        print(f'  {list(route.methods)[0]:6s} {route.path}')

### 2.2 Test the HTTP Layer

`TestClient` from FastAPI runs the app in-process — no server needed.  
This is the right tool for testing HTTP logic (auth, routing, response schema).

In [ ]:
# ── Test app with a mocked LLM — no API key required ──────────────────────
test_app    = create_app(llm=make_mock_llm('Order SO-1001 is Open.'))
test_client = TestClient(test_app)


def test_health_endpoint():
    r = test_client.get('/health')
    assert r.status_code == 200
    assert r.json()['status'] == 'ok'
    print(f'  ✓ GET /health → {r.json()}')


def test_chat_endpoint_returns_reply():
    r = test_client.post('/chat', json={'message': 'What is order SO-1001?', 'thread_id': 'http-test-1'})
    assert r.status_code == 200
    body = r.json()
    assert 'reply' in body
    assert 'thread_id' in body
    assert body['thread_id'] == 'http-test-1'
    print(f'  ✓ POST /chat → reply: "{body["reply"][:60]}"')


def test_chat_endpoint_preserves_thread_id():
    thread = 'my-session-abc'
    r = test_client.post('/chat', json={'message': 'Hello', 'thread_id': thread})
    assert r.json()['thread_id'] == thread
    print(f'  ✓ thread_id preserved: {thread}')


def test_unknown_route_returns_404():
    r = test_client.get('/nonexistent')
    assert r.status_code == 404
    print('  ✓ unknown route → 404')


print('HTTP layer tests:')
test_health_endpoint()
test_chat_endpoint_returns_reply()
test_chat_endpoint_preserves_thread_id()
test_unknown_route_returns_404()
print('All HTTP tests passed.')

---
## Section 3 — Streaming

Streaming sends LLM tokens to the client as they are generated — no waiting for the full response.  
This is essential for any real UI (Fiori, BTP Build Apps, custom chat interfaces).

LangGraph uses **Server-Sent Events (SSE)** via FastAPI's `StreamingResponse`.

```
POST /chat/stream
    │
    │  astream_events()   ← async generator, fires on every event
    │
    ├── event: on_chat_model_stream  → send token chunk
    ├── event: on_tool_start         → send tool name (optional UI update)
    └── event: on_chain_end          → send {"type": "done"}

Client receives:
    data: {"type": "token",      "content": "The "}\n\n
    data: {"type": "token",      "content": "order"}\n\n
    data: {"type": "tool_start", "tool":    "lookup_order"}\n\n
    data: {"type": "token",      "content": "is Open."}\n\n
    data: {"type": "done"}\n\n
```

### 3.1 Streaming Endpoint

We extend the `create_app` factory to include the streaming route.

In [ ]:
import json
from fastapi.responses import StreamingResponse


def create_app_with_streaming(llm=None) -> FastAPI:
    'FastAPI app with both sync /chat and streaming /chat/stream endpoints.'

    _agent = build_agent(llm=llm, checkpointer=MemorySaver())

    app = FastAPI(title='SAP Agent API', version='2.0.0')

    @app.get('/health')
    def health():
        return {'status': 'ok', 'service': 'sap-agent'}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        config = {'configurable': {'thread_id': req.thread_id, 'user_id': req.user_id}}
        result = _agent.invoke(
            {'messages': [HumanMessage(content=req.message)]},
            config=config,
        )
        return ChatResponse(reply=result['messages'][-1].content, thread_id=req.thread_id)

    @app.post('/chat/stream')
    async def chat_stream(req: ChatRequest):
        'Server-Sent Events: streams tokens as they arrive from the LLM.'
        # Use a separate thread_id for streaming to avoid checkpoint conflicts
        config = {'configurable': {'thread_id': req.thread_id + '-s', 'user_id': req.user_id}}

        async def generate():
            async for event in _agent.astream_events(
                {'messages': [HumanMessage(content=req.message)]},
                config=config,
                version='v2',
            ):
                kind = event['event']

                # LLM token chunk
                if kind == 'on_chat_model_stream':
                    chunk = event['data']['chunk'].content
                    if chunk:
                        payload = json.dumps({'type': 'token', 'content': chunk})
                        yield 'data: ' + payload + '\n\n'

                # Tool execution started (optional — useful for UI spinners)
                elif kind == 'on_tool_start':
                    payload = json.dumps({'type': 'tool_start', 'tool': event['name']})
                    yield 'data: ' + payload + '\n\n'

                # Graph finished
                elif kind == 'on_chain_end' and event.get('name') == 'LangGraph':
                    yield 'data: ' + json.dumps({'type': 'done'}) + '\n\n'

        return StreamingResponse(generate(), media_type='text/event-stream')

    return app


streaming_app = create_app_with_streaming()
print('Streaming app created.')
print('Endpoints:')
for route in streaming_app.routes:
    if hasattr(route, 'methods'):
        print(f'  {list(route.methods)[0]:6s} {route.path}')

### 3.2 Streaming Demo

We demonstrate the streaming pattern using LangGraph's synchronous `stream()` API,  
which has the same event structure as `astream_events` and runs inline in the notebook.

In [ ]:
# ── Synchronous streaming demo (same event types as the async SSE endpoint) ─
demo_agent = build_agent()   # requires OPENAI_API_KEY

if os.getenv('OPENAI_API_KEY'):
    print('Streaming demo (stream_mode="messages"):')
    for chunk, metadata in demo_agent.stream(
        {'messages': [HumanMessage(content='What is order SO-1001?')]},
        stream_mode='messages',
    ):
        if hasattr(chunk, 'content') and chunk.content:
            print(chunk.content, end='', flush=True)
    print()  # newline after stream
else:
    print('No OPENAI_API_KEY — showing stream event structure instead:')
    # Simulate what the SSE endpoint would send
    simulated_events = [
        {'type': 'token',      'content': 'Order '},
        {'type': 'tool_start', 'tool':    'lookup_order'},
        {'type': 'token',      'content': 'SO-1001 belongs to ACME Corp'},
        {'type': 'token',      'content': ' and is currently Open.'},
        {'type': 'done'},
    ]
    for ev in simulated_events:
        print('data: ' + json.dumps(ev))

print()
print('In production, the POST /chat/stream endpoint sends these events as SSE.')
print('Client-side (JavaScript): const source = new EventSource("/chat/stream")')

---
## Section 4 — AI Core Credential Wiring

So far we have used `ChatOpenAI` directly. On SAP BTP, LLMs are accessed through  
**SAP AI Core → Generative AI Hub** using `gen_ai_hub` (from `sap-ai-sdk-gen`).

### How `gen_ai_hub` picks up credentials automatically

The SDK checks three locations in order:

```
1. Individual env vars  AICORE_CLIENT_ID + AICORE_CLIENT_SECRET + AICORE_AUTH_URL + AICORE_BASE_URL
2. JSON env var         AICORE_SERVICE_KEY='{"clientid":"...","clientsecret":"...","url":"..."}'
3. Config file          ~/.aicore/config.json
```

On Cloud Foundry you inject option 2 via `cf set-env` or `manifest.yml`:  
```bash
cf set-env sap-agent-api AICORE_SERVICE_KEY '{"clientid":"sb-...", ...}'
cf restage sap-agent-api
```

### Credential-aware `get_llm()` function

This function is the single place in the codebase that decides which LLM to use.  
Copy it into your `main.py` for deployment.

In [ ]:
def get_llm():
    'Return a chat LLM — uses SAP AI Core if credentials are present, else OpenAI.'

    # ── Check which credential format is present ───────────────────────────
    has_aicore = bool(
        os.getenv('AICORE_CLIENT_ID')                              or  # Format A
        os.getenv('AICORE_SERVICE_KEY')                            or  # Format B
        os.path.exists(os.path.expanduser('~/.aicore/config.json'))    # Format C
    )

    if has_aicore:
        try:
            from gen_ai_hub.proxy.langchain.openai import ChatOpenAI as AICoreChat
            from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

            proxy_client = get_proxy_client('gen-ai-hub')
            print('LLM: SAP AI Core (gen_ai_hub) — gpt-4o')
            return AICoreChat(
                proxy_model_name='gpt-4o',
                proxy_client=proxy_client,
                temperature=0,
                max_tokens=2048,
            )
        except ImportError:
            print('sap-ai-sdk-gen not installed — falling back to OpenAI')

    # ── Fallback: direct OpenAI (local dev) ───────────────────────────────
    print('LLM: OpenAI (local dev mode) — gpt-4o-mini')
    return ChatOpenAI(model='gpt-4o-mini', temperature=0)


llm = get_llm()
print(f'LLM class: {type(llm).__name__}')

# ── Rebuild the agent with the credential-aware LLM ───────────────────────
agent = build_agent(llm=llm, checkpointer=MemorySaver())
print('Agent rebuilt with get_llm().')

---
## Section 5 — SAP Workflow Tools

Now we upgrade the mock tools to the actual patterns used in production SAP integrations.

### Two integration patterns

| Tool | Backend | Auth | Library |
|------|---------|------|---------|
| `search_sap_kb` | SAP HANA Cloud Vector Engine | HANA `dbapi` | `langchain-hana` (official SAP) |
| `get_sales_orders` | SAP S/4HANA or CAP OData V4 | XSUAA OAuth2 bearer token | `requests` |

Both are mocked below — the structure is identical to production; only the connection lines change.

### 5.1 HANA Vector Search Tool

In [ ]:
from langchain_core.documents import Document


# ── Production replacement: ──────────────────────────────────────────────
#   from langchain_hana import HanaDB, HanaInternalEmbeddings
#   import hdbcli.dbapi as dbapi
#   conn = dbapi.connect(address=os.getenv('HANA_HOST'), port=443,
#                        user=os.getenv('HANA_USER'), password=os.getenv('HANA_PASSWORD'))
#   _vectorstore = HanaDB(
#       connection=conn,
#       embedding=HanaInternalEmbeddings(conn),   # uses HANA native VECTOR_EMBEDDING()
#       table_name='SAP_AGENT_KB',
#   )

class MockHanaDB:
    'Drop-in mock for langchain-hana HanaDB.'

    def __init__(self, docs: list):
        self.docs = docs

    def similarity_search(self, query: str, k: int = 2) -> list:
        words = set(query.lower().split())
        scored = [(len(words & set(d.page_content.lower().split())), d) for d in self.docs]
        scored.sort(key=lambda x: x[0], reverse=True)
        return [d for _, d in scored[:k]]


_vectorstore = MockHanaDB([
    Document(page_content='SO-1001: ACME Corp, 12,500 EUR, Open. Created 2024-01-15.',
             metadata={'source': 'S4HANA', 'type': 'sales_order'}),
    Document(page_content='SO-1002: SAP SE, 87,000 EUR, Delivered. Shipped 2024-03-02.',
             metadata={'source': 'S4HANA', 'type': 'sales_order'}),
    Document(page_content='TKT-001: System outage BTP tenant, high priority, ops team, ETA 2h.',
             metadata={'source': 'ServiceCloud', 'type': 'ticket'}),
    Document(page_content='TKT-002: Performance degradation in AI Core, medium priority.',
             metadata={'source': 'ServiceCloud', 'type': 'ticket'}),
    Document(page_content='SAP BTP AI Core pricing: enterprise tier includes HANA Cloud and Gen AI Hub.',
             metadata={'source': 'BTPCatalog', 'type': 'product'}),
])


@tool
def search_sap_kb(query: str) -> str:
    'Search the SAP HANA knowledge base for orders, tickets, and product info.'
    docs = _vectorstore.similarity_search(query, k=2)
    if not docs:
        return 'No relevant SAP data found.'
    return '\n\n'.join(
        f'[{d.metadata["source"]}|{d.metadata["type"]}] {d.page_content}'
        for d in docs
    )


# ── Quick smoke test ───────────────────────────────────────────────────────
print(search_sap_kb.invoke({'query': 'ACME sales order status'}))
print('---')
print(search_sap_kb.invoke({'query': 'high priority ticket outage'}))

### 5.2 CAP / S/4HANA OData GET Tool

Any HTTP endpoint — CAP OData, S/4HANA, SAP Integration Suite — becomes a tool by wrapping it with `@tool`.  
Auth uses XSUAA OAuth2 (a bearer token from the XSUAA token endpoint).

In [ ]:
# ── Production replacement (OAuth2 bearer token flow): ───────────────────
#   import requests
#
#   def get_xsuaa_token() -> str:
#       resp = requests.post(
#           url=os.getenv('XSUAA_URL') + '/oauth/token',
#           data={'grant_type': 'client_credentials'},
#           auth=(os.getenv('XSUAA_CLIENT_ID'), os.getenv('XSUAA_CLIENT_SECRET')),
#       )
#       return resp.json()['access_token']
#
#   @tool
#   def get_sales_orders(customer_id: str) -> str:
#       token = get_xsuaa_token()
#       url   = os.getenv('S4_BASE_URL') + '/sap/opu/odata/sap/API_SALES_ORDER_SRV/A_SalesOrder'
#       resp  = requests.get(
#           url,
#           params={'$filter': f"SoldToParty eq '{customer_id}'", '$top': '5'},
#           headers={'Authorization': f'Bearer {token}', 'Accept': 'application/json'},
#       )
#       return json.dumps(resp.json().get('d', {}).get('results', []))

@tool
def get_sales_orders(customer_id: str) -> str:
    'Fetch open sales orders for a customer from S/4HANA OData API. Use customer short name (ACME, SAPSE).'
    # Mocked response — same structure as the real OData response
    mock_data = {
        'ACME':  '[{"SalesOrder": "SO-1001", "NetAmount": "12500", "Currency": "EUR", "OverallSDProcessStatus": "A"}]',
        'SAPSE': '[{"SalesOrder": "SO-1002", "NetAmount": "87000", "Currency": "EUR", "OverallSDProcessStatus": "C"}]',
    }
    result = mock_data.get(customer_id.upper())
    if result:
        return f'Open orders for {customer_id}: {result}'
    return f'No orders found for customer {customer_id}.'


# ── Upgrade SAP_TOOLS to the full production-ready set ────────────────────
SAP_TOOLS_V2 = [search_sap_kb, get_sales_orders, lookup_order]

print('SAP_TOOLS_V2:', [t.name for t in SAP_TOOLS_V2])
print()
print(get_sales_orders.invoke({'customer_id': 'ACME'}))

In [ ]:
# ── Rebuild with upgraded tools ────────────────────────────────────────────
agent = build_agent(llm=get_llm(), tools=SAP_TOOLS_V2, checkpointer=MemorySaver())
app   = create_app_with_streaming(llm=get_llm())

# Quick live test (requires API key)
if os.getenv('OPENAI_API_KEY'):
    r = agent.invoke({
        'messages': [HumanMessage(content='What are ACME Corp open orders, and any related context from the knowledge base?')]
    })
    print('Agent response:')
    for msg in r['messages']:
        role = msg.__class__.__name__.replace('Message', '')
        body = msg.content or str(getattr(msg, 'tool_calls', ''))
        print(f'  [{role:12s}] {body[:150]}')
else:
    print('No OPENAI_API_KEY — agent compiled with SAP_TOOLS_V2, ready to run.')

---
## Section 6 — Cloud Foundry Deployment

We now have everything we need. This section assembles the deployment package.

### 6.1 File Structure

```
my-sap-agent/
├── main.py            ← FastAPI entrypoint (shown below)
├── agent.py           ← build_agent(), tools (Sections 1–5)
├── manifest.yml       ← Cloud Foundry app descriptor
├── runtime.txt        ← Python version pin
└── requirements.txt   ← Python dependencies
```

### 6.2 Deployment Files

In [ ]:
# ── manifest.yml ───────────────────────────────────────────────────────────
MANIFEST_YML = '''\
applications:
  - name: sap-agent-api
    memory: 512M
    disk_quota: 2G
    instances: 1
    buildpack: python_buildpack
    # CF injects PORT automatically; gunicorn+uvicorn is the recommended stack
    command: gunicorn -w 1 -k uvicorn.workers.UvicornWorker main:app --bind 0.0.0.0:${PORT:-8000}
    random-route: true
    env:
      # Paste your AI Core service key JSON as a single-line string
      # Get it from: SAP BTP Cockpit → AI Core service instance → Service Keys
      AICORE_SERVICE_KEY: '{"clientid":"sb-...","clientsecret":"...","url":"https://...","serviceurls":{"AI_API_URL":"https://api.ai..."},"appname":"..."}'
      # Optional: Langfuse observability (Section 8 of Notebook 2)
      # LANGFUSE_PUBLIC_KEY: "pk-lf-..."
      # LANGFUSE_SECRET_KEY: "sk-lf-..."
      # LANGFUSE_HOST:       "https://your-langfuse-cf-app.cfapps.eu10.hana.ondemand.com"
'''

# ── runtime.txt ────────────────────────────────────────────────────────────
RUNTIME_TXT = 'python-3.11.x'

# ── requirements.txt ───────────────────────────────────────────────────────
REQUIREMENTS_TXT = '''\
langgraph>=1.0.0
langchain-core>=0.3.0
langchain-openai>=1.0.0
fastapi>=0.115.0
uvicorn>=0.30.0
gunicorn>=22.0.0
pydantic>=2.0.0
sap-ai-sdk-gen[all]>=6.0.0
httpx>=0.27.0
'''

print('=== manifest.yml ===')
print(MANIFEST_YML)
print('=== runtime.txt ===')
print(RUNTIME_TXT)
print('\n=== requirements.txt ===')
print(REQUIREMENTS_TXT)

### 6.3 Complete `main.py`

This is the entrypoint FastAPI file the CF buildpack will look for.  
It combines everything from Sections 2–5 into one production-ready module.

In [ ]:
MAIN_PY = '''\
# main.py — SAP Agent API entrypoint for Cloud Foundry
import os
import json
from contextlib import asynccontextmanager
from typing import Optional

from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

from agent import build_agent  # your agent module


# ── Credential-aware LLM factory ──────────────────────────────────────────
def get_llm():
    has_aicore = bool(
        os.getenv("AICORE_CLIENT_ID") or
        os.getenv("AICORE_SERVICE_KEY") or
        os.path.exists(os.path.expanduser("~/.aicore/config.json"))
    )
    if has_aicore:
        from gen_ai_hub.proxy.langchain.openai import ChatOpenAI as AICoreChat
        from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client
        return AICoreChat(proxy_model_name="gpt-4o",
                          proxy_client=get_proxy_client("gen-ai-hub"),
                          temperature=0)
    return ChatOpenAI(model="gpt-4o-mini", temperature=0)


# ── App with lifespan (single agent instance, shared across requests) ─────
@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.agent = build_agent(llm=get_llm(), checkpointer=MemorySaver())
    print("Agent initialised.")
    yield


app = FastAPI(title="SAP Agent API", version="1.0.0", lifespan=lifespan)


class ChatRequest(BaseModel):
    message:   str
    thread_id: Optional[str] = "default"


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/chat")
def chat(req: ChatRequest):
    config = {"configurable": {"thread_id": req.thread_id}}
    result = app.state.agent.invoke(
        {"messages": [HumanMessage(content=req.message)]}, config=config
    )
    return {"reply": result["messages"][-1].content, "thread_id": req.thread_id}


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    config = {"configurable": {"thread_id": req.thread_id + "-s"}}

    async def generate():
        async for event in app.state.agent.astream_events(
            {"messages": [HumanMessage(content=req.message)]},
            config=config, version="v2",
        ):
            kind = event["event"]
            if kind == "on_chat_model_stream":
                chunk = event["data"]["chunk"].content
                if chunk:
                    yield "data: " + json.dumps({"type": "token", "content": chunk}) + "\\n\\n"
            elif kind == "on_chain_end" and event.get("name") == "LangGraph":
                yield "data: " + json.dumps({"type": "done"}) + "\\n\\n"

    return StreamingResponse(generate(), media_type="text/event-stream")
'''

print(MAIN_PY)

### 6.4 Deploy to Cloud Foundry

```bash
# 1. Save the files from the cells above
# main.py, agent.py, manifest.yml, runtime.txt, requirements.txt

# 2. Log in to your BTP CF space
cf login -a https://api.cf.<region>.hana.ondemand.com

# 3. (Optional) set the AI Core key as env var — or embed it in manifest.yml
cf set-env sap-agent-api AICORE_SERVICE_KEY '{"clientid":"sb-...", ...}'

# 4. Push!
cf push

# 5. Check logs
cf logs sap-agent-api --recent

# 6. Test the deployed endpoint
curl https://<your-cf-route>/health
# → {"status": "ok"}

curl -X POST https://<your-cf-route>/chat \
  -H 'Content-Type: application/json' \
  -d '{"message": "What is order SO-1001?", "thread_id": "prod-001"}'
```

**CF memory note:** Start with `512M`. If you see `out of memory` in logs, bump to `1G`.  
LangGraph's graph compilation happens at startup — it's a one-time cost.

**Gunicorn workers:** Use `-w 1` (one worker). Multiple workers don't share the in-memory  
checkpointer — you'd get different conversation histories per request.  
For multi-worker setups, use an external checkpointer (Redis, Postgres, or HANA).

---
## Section 7 — MCP Tools

The **Model Context Protocol (MCP)** is a standardised way for LLMs to discover and call external tools.  
LangGraph agents can connect to any MCP server using `langchain-mcp-adapters`.

**SAP MCP server for CAP** (`@cap-js/mcp-server`) exposes:
- **Model search** — fuzzy search over CDS entity definitions, services, endpoints, annotations
- **Documentation search** — semantic search over CAP docs

> Note: the CAP MCP server is **developer-time tooling** — it helps an LLM understand a CAP project's  
> schema and docs, not make live OData calls. For runtime API calls, use the `@tool`-wrapped OData  
> pattern from Section 5. Both can coexist in the same agent.

```
LangGraph agent
  │
  ├─ SAP_TOOLS_V2     ← from Section 5 (HANA, OData)
  │
  └─ MCP tools        ← from langchain-mcp-adapters
       │
       └─ CAP MCP server (npx @cap-js/mcp-server)
              ├─ search_cds_models(query)
              └─ search_cap_docs(query)
```

In [ ]:
try:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    MCP_OK = True
    print('langchain-mcp-adapters installed.')
except ImportError:
    MCP_OK = False
    print('Not installed — run: pip install langchain-mcp-adapters')
    print('Integration pattern is shown below regardless.')

In [ ]:
import asyncio

# ── Pattern: connect to MCP servers and merge tools into the agent ─────────
#
# MultiServerMCPClient is an async context manager.
# In Jupyter: use `await` directly. In a script: use asyncio.run().

MCP_CONFIG = {
    # CAP MCP server — exposes CDS model search and CAP documentation search
    # Prerequisite: Node.js + npx installed
    'cap': {
        'command':   'npx',
        'args':      ['-y', '@cap-js/mcp-server'],
        'transport': 'stdio',
    },
    # Example: a custom SAP data MCP server (SSE transport)
    # 'sap-data': {
    #     'url':       'http://localhost:8001/sse',
    #     'transport': 'sse',
    # },
}


async def build_agent_with_mcp(base_tools=None):
    'Build the agent with MCP tools merged alongside the SAP tools.'
    if base_tools is None:
        base_tools = SAP_TOOLS_V2

    async with MultiServerMCPClient(MCP_CONFIG) as client:
        mcp_tools = await client.get_tools()
        print(f'MCP tools discovered ({len(mcp_tools)}):')
        for t in mcp_tools:
            print(f'  {t.name}: {t.description[:70]}')

        all_tools = base_tools + mcp_tools
        agent = build_agent(tools=all_tools, checkpointer=MemorySaver())
        print(f'\nAgent built with {len(all_tools)} tools total.')
        return agent


# ── Run ────────────────────────────────────────────────────────────────────
if MCP_OK:
    print('To build the agent with MCP tools:')
    print('  In Jupyter:  agent = await build_agent_with_mcp()')
    print('  In a script: agent = asyncio.run(build_agent_with_mcp())')
    print()
    print('Requires: Node.js + npx installed for the CAP MCP server.')
    print('Or replace MCP_CONFIG with any other MCP server URL.')
else:
    print('Install langchain-mcp-adapters to use MCP tools.')

print()
print('Without MCP: the agent works fine with SAP_TOOLS_V2 only.')
print('MCP adds discoverability of CAP model schemas and docs at query time.')

---
## Section 8 — Joule Studio Integration

**SAP Joule Studio Agent Builder** reached General Availability in late 2025.  
It lets you call any HTTP endpoint as a Joule skill — including the FastAPI app we just deployed.

### Architecture

```
┌─────────────────────────────────────────────────────────────────┐
│                       SAP BTP Subaccount                         │
│                                                                   │
│  ┌──────────────┐   HTTP via BTP Destination   ┌──────────────┐  │
│  │ SAP Joule    │ ───────────────────────────► │  FastAPI CF  │  │
│  │ Studio Skill │                              │  POST /chat  │  │
│  └──────────────┘                              └──────┬───────┘  │
│                                                       │           │
│                                               ┌───────▼───────┐  │
│                                               │  LangGraph    │  │
│                                               │  Agent        │  │
│                                               └───────┬───────┘  │
│                                                       │           │
│                              ┌────────────────────────┤           │
│                              │                        │           │
│                    ┌─────────▼──────┐      ┌─────────▼────────┐  │
│                    │  AI Core /     │      │  HANA Cloud /    │  │
│                    │  Gen AI Hub    │      │  S/4HANA OData   │  │
│                    └────────────────┘      └──────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

### Integration steps

1. **Deploy** the FastAPI agent to CF (Section 6) — note the CF route URL
2. **Create a BTP Destination** pointing to the CF app URL (with auth if needed)
3. **Create a Joule Skill** in Joule Studio Agent Builder:
   - Skill type: HTTP Action
   - Destination: the one you created
   - Path: `/chat`
   - Method: POST
   - Body schema: `{"message": "...", "thread_id": "..."}`
4. **Publish** the skill to Joule Studio

The key insight: **the same FastAPI endpoint** that you call with `curl` or from a browser  
is the exact same endpoint Joule Studio calls. No separate integration code needed.

### What Joule adds

| Without Joule | With Joule Studio |
|---------------|-------------------|
| Direct HTTP calls to your CF app | Discoverable SAP Joule skill |
| Users need to know the endpoint URL | Users interact through the Joule chat UI |
| No SAP identity propagation | XSUAA user context can be forwarded |
| Manual access management | Managed via BTP role collections |

In [ ]:
# ── Joule Studio sends the same JSON body as any other client ─────────────
# You can simulate a Joule call by using TestClient:

joule_client = TestClient(create_app_with_streaming(llm=make_mock_llm('Order SO-1001 is Open.')))

# Joule Studio sends a POST /chat with the user's message and a session ID
joule_request = {
    'message':   'What is the status of order SO-1001?',
    'thread_id': 'joule-session-user-12345',    # Joule provides a session ID per user
}

r = joule_client.post('/chat', json=joule_request)
print('Simulated Joule Studio call:')
print(f'  Status:    {r.status_code}')
print(f'  Reply:     {r.json()["reply"]}')
print(f'  Thread ID: {r.json()["thread_id"]}')
print()
print('The deployed CF app responds identically to Joule, curl, and direct HTTP calls.')

---
## Summary & What You Built

### The app you assembled across this notebook

```
my-sap-agent/
├── agent.py        ← build_agent(), SAP tools (HANA + OData)
├── main.py         ← FastAPI: /health, /chat, /chat/stream
├── manifest.yml    ← CF: memory, buildpack, gunicorn command, env vars
├── runtime.txt     ← python-3.11.x
└── requirements.txt
```

### What each section contributed

| Section | What it added |
|---------|---------------|
| 1 — Testing | pytest patterns · mock LLM · node unit tests · graph integration tests |
| 2 — FastAPI | App factory · `/chat` · `/health` · Pydantic models · TestClient |
| 3 — Streaming | `/chat/stream` · `astream_events` · SSE · token/tool_start/done events |
| 4 — AI Core | `get_llm()` · `AICORE_SERVICE_KEY` · `gen_ai_hub` auto-detection |
| 5 — SAP Tools | `MockHanaDB` → `HanaDB` · CAP/S4 OData `@tool` · `SAP_TOOLS_V2` |
| 6 — CF Deploy | `manifest.yml` · `runtime.txt` · `requirements.txt` · complete `main.py` |
| 7 — MCP | `langchain-mcp-adapters` · CAP MCP server · merge with SAP tools |
| 8 — Joule | Architecture · BTP Destination pattern · same endpoint, zero extra code |

### The three-notebook learning arc

| Notebook | Focus |
|----------|-------|
| 1 — Fundamentals | State · Nodes · Edges · ReAct · Memory · HITL · gen_ai_hub intro |
| 2 — Intermediate | Supervisor · Subgraphs · Send API · HANA RAG · Long-term memory · Reflection · Langfuse |
| 3 — Deployment | Testing · FastAPI · Streaming · AI Core credentials · SAP tools · CF · MCP · Joule |

### Key references

- [QuickLaunch Part 6 — FastAPI REST API](https://community.sap.com/t5/technology-blog-posts-by-sap/sap-ai-core-agent-quicklaunch-series-part-6-converting-the-ai-agent-into-a/ba-p/14121055)
- [QuickLaunch Part 8 — Cloud Foundry Deploy](https://community.sap.com/t5/technology-blog-posts-by-sap/sap-ai-core-agent-quicklaunch-series-part-8-deploying-to-cloud-foundry/ba-p/14127673)
- [Kicking Off LangGraph Agents from SAP Joule Studio](https://community.sap.com/t5/enterprise-resource-planning-blog-posts-by-sap/kicking-off-langgraph-agents-from-sap-joule-studio/ba-p/14174912)
- [langchain-mcp-adapters](https://github.com/langchain-ai/langchain-mcp-adapters)
- [cap-js/mcp-server](https://github.com/cap-js/mcp-server)
- [LangGraph streaming docs](https://langchain-ai.github.io/langgraph/concepts/streaming/)
- [FastAPI streaming + LangGraph](https://python.langchain.com/docs/how_to/streaming_llm/)